# Laboratorio 5: Modelo de datos avanzado
### Proyecto: 
Análisis del Rendimiento Académico
 
###  Integrantes:
- Lagos Ratto, Norelyz
- Rumay Avedaño, Meylin 
- Vasquez Llallahui, Maylhy
- León Capuñay, Pierre


## 1. Subir Archivos 

## 2. Definir esquema esperado con StructType

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Region_Type", StringType(), True),
    StructField("Family_Size", IntegerType(), True),
    StructField("Parent_Education", StringType(), True),
    StructField("Family_Income_Level", StringType(), True),
    StructField("Internet_Quality", DoubleType(), True),
    StructField("Study_Space_Quality", DoubleType(), True),
    StructField("Previous_GPA", DoubleType(), True),
    StructField("Number_of_Failed_Courses", IntegerType(), True),
    StructField("Total_Credits_Earned", IntegerType(), True),
    StructField("Weekly_Study_Hours", DoubleType(), True),
    StructField("Attendance_Rate", DoubleType(), True),
    StructField("Library_Visits_Per_Month", IntegerType(), True),
    StructField("Extracurricular_Hours", DoubleType(), True),
    StructField("Sleep_Hours", DoubleType(), True),
    StructField("Social_Media_Usage_Hours", DoubleType(), True),
    StructField("Stress_Level", DoubleType(), True),
    StructField("Motivation_Score", DoubleType(), True),
    StructField("Self_Efficacy_Score", DoubleType(), True),
    StructField("Midterm_Mark", DoubleType(), True),
    StructField("Final_Exam_Score", DoubleType(), True),
])


## 3. Leer y visualizar archivo json 

In [0]:
from pyspark.sql.functions import col

df_estudiantes = (
    spark.read
    .format("json")
    .schema(schema)
    .load("/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion05/")
    .withColumn("archivo_origen", col("_metadata.file_path"))
)

In [0]:
display(df_estudiantes)

## 4. Leer archivo jason y mapear inconsistencias

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Region_Type", StringType(), True),
    StructField("Family_Size", IntegerType(), True),
    StructField("Parent_Education", StringType(), True),
    StructField("Family_Income_Level", StringType(), True),
    StructField("Internet_Quality", DoubleType(), True),
    StructField("Study_Space_Quality", DoubleType(), True),
    StructField("Previous_GPA", DoubleType(), True),
    StructField("Number_of_Failed_Courses", IntegerType(), True),
    StructField("Total_Credits_Earned", IntegerType(), True),
    StructField("Weekly_Study_Hours", DoubleType(), True),
    StructField("Attendance_Rate", DoubleType(), True),
    StructField("Library_Visits_Per_Month", IntegerType(), True),
    StructField("Extracurricular_Hours", DoubleType(), True),
    StructField("Sleep_Hours", DoubleType(), True),
    StructField("Social_Media_Usage_Hours", DoubleType(), True),
    StructField("Stress_Level", DoubleType(), True),
    StructField("Motivation_Score", DoubleType(), True),
    StructField("Self_Efficacy_Score", DoubleType(), True),
    StructField("Midterm_Mark", DoubleType(), True),
    StructField("Final_Exam_Score", DoubleType(), True),
    StructField("_rescued_data", StringType(), True)
])


In [0]:
from pyspark.sql.functions import col

try:
    df_estudiantes = (
    spark.read
    .format("json")
    .schema(schema)
    .option("multiLine", "true")
    .option("columnNameOfCorruptRecord", "_rescued_data")
    .load("/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion05/")
    .withColumn("archivo_origen", col("_metadata.file_path"))
    )

    print("Lectura exitosa")

except Exception as e:
    print("Error en la lectura:", str(e))


In [0]:
display(df_estudiantes)

## 5. Verificar archivos de origen

In [0]:
display(df_estudiantes.select("archivo_origen").distinct())

## 6. Separación de datos válidos e inválidos

In [0]:
# Filtrar por _rescued_data (NULL = válido, NOT NULL = inválido)
df_validos = df_estudiantes.filter(col("_rescued_data").isNull())

df_invalidos = df_estudiantes.filter(col("_rescued_data").isNotNull())


## 7. Validación visual

In [0]:
display(df_validos)
display(df_invalidos)

## 7. Conteo de registros

In [0]:
# Query saved tables to avoid _rescued_data restriction
validos_count = spark.table("proyecto_estudiantes.silver.estudiantes_validos").count()
invalidos_count = spark.table("proyecto_estudiantes.silver.estudiantes_invalidos").count()

print("Registros válidos:", validos_count)
print("Registros inválidos:", invalidos_count)

## 8. Creamos el esquema SILVER

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS proyecto_estudiantes.silver;

## 9. Guardar en tablas Delta

In [0]:
# Drop metadata columns before writing
df_validos_clean = df_validos.drop("_rescued_data") if "_rescued_data" in df_validos.columns else df_validos
df_invalidos_clean = df_invalidos.drop("_rescued_data") if "_rescued_data" in df_invalidos.columns else df_invalidos

df_validos_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "proyecto_estudiantes.silver.estudiantes_validos"
)

df_invalidos_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "proyecto_estudiantes.silver.estudiantes_invalidos"
)

## 10. Verificación final

In [0]:
%sql
select * from proyecto_estudiantes.silver.estudiantes_validos

In [0]:
%sql
select * from proyecto_estudiantes.silver.estudiantes_invalidos